# 02 - Data Cleaning
Clean missing values, fix inconsistent names, and save cleaned CSVs for the next notebooks.

In [1]:
import pandas as pd
import numpy as np

matches = pd.read_csv('data/matches.csv')
deliveries = pd.read_csv('data/deliveries.csv')

## 1. Missing values

In [2]:
print(matches.isnull().sum())
print()
print(deliveries.isnull().sum())

id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64

match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64


## 2. Duplicates

In [4]:
print('Duplicate matches   :', matches.duplicated().sum())
print('Duplicate deliveries :', deliveries.duplicated().sum())
matches = matches.drop_duplicates()
deliveries = deliveries.drop_duplicates()

Duplicate matches   : 0
Duplicate deliveries : 0


## 3. Standardise team names
Franchises were renamed over the years - map old names to the current one.

In [5]:
team_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Gujarat Lions': 'Gujarat Titans',
}
for col in ['team1', 'team2', 'toss_winner', 'winner']:
    matches[col] = matches[col].replace(team_map)
for col in ['batting_team', 'bowling_team']:
    deliveries[col] = deliveries[col].replace(team_map)

print(sorted(matches['team1'].unique()))

['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


> Note: mapping Deccan Chargers -> Sunrisers and Gujarat Lions -> Gujarat Titans is debatable (different franchises).
> Remove those two lines from `team_map` if you want them kept separate.

## 4. Fill missing values

In [7]:
matches['city'] = matches['city'].fillna('Unknown')
matches['winner'] = matches['winner'].fillna('No Result')
matches['player_of_match'] = matches['player_of_match'].fillna('Not Awarded')
matches['result_margin'] = matches['result_margin'].fillna(0)
matches['method'] = matches['method'].fillna('Normal')

deliveries['extras_type'] = deliveries['extras_type'].fillna('None')
deliveries['player_dismissed'] = deliveries['player_dismissed'].fillna('None')
deliveries['dismissal_kind'] = deliveries['dismissal_kind'].fillna('None')
deliveries['fielder'] = deliveries['fielder'].fillna('None')

## 5. Fix data types & fix season labels

In [8]:
matches['date'] = pd.to_datetime(matches['date'])

# '2007/08' -> 2008, '2009/10' -> 2010, '2020/21' -> 2020 ; others stay as-is
season_map = {'2007/08': '2008', '2009/10': '2010', '2020/21': '2020'}
matches['season'] = matches['season'].astype(str).replace(season_map).astype(int)
print(sorted(matches['season'].unique()))

[np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


## 6. Sanity checks

In [9]:
print(matches.isnull().sum().sum(), deliveries.isnull().sum().sum())
print('Match ids in deliveries but not in matches:',
      (~deliveries['match_id'].isin(matches['id'])).sum())

6 0
Match ids in deliveries but not in matches: 0


## 7. Save cleaned data

In [ ]:
matches.to_csv('data/matches_clean.csv', index=False)
deliveries.to_csv('data/deliveries_clean.csv', index=False)
print('saved')